In [ ]:
pip install -r requirements.txt

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
from cheb_ar.solvers.cheb_ar import *

In [ ]:
# ---------------- parameters (same as your snippet) ----------------
w_a = 25.338776456203686
w_b = 2 * w_a
kappa_b = 5 / 10.4
E_J = 37 * 2 * np.pi
phi_a, phi_b = 0.11, 0.204
epsilon_p = 0.1
g  = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
kappa_2 = 4 * g**2 / kappa_b
kappa_1 = 0.005 * kappa_2
n_a, n_b = 30, 11
dims = (n_a, n_b)
alpha_sq = 8.5
epsilon_d = 2 * alpha_sq * g2

N = n_a * n_b
T_block = 2 * jnp.pi / w_a
tsave = jnp.array([0.0, T_block])
method = dq.method.Tsit5(rtol=1e-9, atol=1e-10)
opts = dq.Options(assume_hermitian=False)

In [ ]:
from cheb_ar.models.ats import build_ats_hamiltonian_interaction

In [ ]:
H_I, jump_ops_I, jump_ops_LdL_I, output_phase, V, T_block, params  = build_ats_hamiltonian_interaction(n_a = n_a, n_b = n_b, alpha_sq = alpha_sq, epsilon_p = epsilon_p)
solver = ChebAr(H_I, jump_ops_I, T_block, jump_ops_LdL=jump_ops_LdL_I, dims=dims, output_phase=output_phase)
m_arnoldi_0 = 120
x0 = solver.make_x0(seed=0)
_, _, ritz_vals = solver.first_estimation(x0, m_arnoldi=m_arnoldi_0)
margin = 1e-2
solver.setup_chebyshev(ritz_vals, margin=margin)
warm_start = False
m_arnoldi = 120
Q, H, mu_list = solver.arnoldi_hessenberg(x0, solver.chebyshev_filter, m_arnoldi, warm_start = warm_start)


In [ ]:
plt.scatter(range(m_arnoldi), np.abs(1-np.real(mu_list)))
plt.xlabel("Arnoldi iteration")
plt.ylabel(r"$1-|\mathrm{Re}(\mu)|$")
plt.yscale("log")

In [ ]:
solver.rate_from_mu(mu_list[-1])

In [ ]:
x_ritz, rho_ritz = solver.ritz_vector(Q, H, m_arnoldi, target=mu_list[-1])
theta = np.linspace(0,2*np.pi, 100)

fig ,ax = plt.subplots()
dq.plot.wigner(dq.ptrace(rho_ritz,0), ax = ax)
ax.plot(np.sqrt(alpha_sq/2) * np.cos(theta), np.sqrt(alpha_sq/2) * np.sin(theta), color='r', 
        linewidth = 1, linestyle = '--')
plt.show()

In [ ]:
res = solver.residual_check(x_ritz)
res